In [ ]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
from pandas.core.dtypes.common import is_numeric_dtype
import ast

In [ ]:
customer_data = pd.read_csv('../data/task02/customer_data_collection.csv')
product_recommendation_data = pd.read_csv('../data/task02/product_recommendation_data.csv')

print(f'Customer Data shape:\n{customer_data.shape}')
print(f"Product Data shape:\n{product_recommendation_data.shape}")

display(customer_data.head())
display(product_recommendation_data.head())

In [ ]:
def remove_unnamed_columns(df):
    return df.loc[:, ~df.columns.str.contains('^Unnamed')]

def view_shape(df, name):
    return f'{name} shape:\n{df.shape}'

customer_data = remove_unnamed_columns(customer_data)
product_recommendation_data = remove_unnamed_columns(product_recommendation_data)

print(view_shape(customer_data, 'Customer Data'))
print(view_shape(product_recommendation_data, 'Product Recommendation Data'))


In [ ]:
print(customer_data.head())
print(customer_data.info())

In [ ]:
print(product_recommendation_data.head())
print(product_recommendation_data.info())

In [ ]:
# Price column from int to float
product_recommendation_data['Price'] = product_recommendation_data['Price'].astype(float)

print(product_recommendation_data.head())


In [ ]:
# Turn headers into snake case

def headers_to_snake_case(df):
    df = df.copy()

    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip('_')
    )
    return df

customer_data = headers_to_snake_case(customer_data)
product_recommendation_data = headers_to_snake_case(product_recommendation_data)

print(customer_data.columns)
print(product_recommendation_data.columns)


In [ ]:
# Check for missing values

def check_missing_values(df, name):
    return f'Missing values in {name}:\n{df.isna().sum()}\n'

print(check_missing_values(customer_data, 'Customer Data'))
print(check_missing_values(product_recommendation_data, 'Product Recommendation Data'))



In [ ]:
# Check for duplicate rows

def check_duplicate_rows(df):
    if df.duplicated().sum() > 0:
        return df[df.duplicated(keep=False)]

    return 'No duplicate rows found!'

print(check_duplicate_rows(customer_data))
print(check_duplicate_rows(product_recommendation_data))

In [ ]:
# Check Invalid Values

def check_values_below_zero(df):

    for col_name in df.columns:
        if is_numeric_dtype(df[col_name]):
            invalid_value = df.loc[
                (df[col_name] < 0) |
                (df[col_name].isna())
            ]
            print(f'Below zero values in "{col_name}": {len(invalid_value)}')

    return f'All Columns checked'


def check_ratings(df, col_name):
    invalid_values = df.loc[
        (df[col_name] < 0) |
        (df[col_name] > 5) |
        (df[col_name].isna())
    ]

    if not invalid_values.empty:
        return f'Invalid ratings in "{col_name}": {len(invalid_values)}'

    return 'No invalid ratings found!'


invalid_product_recommendation_data = product_recommendation_data.loc[
    (product_recommendation_data['probability_of_recommendation'] < 0) |
    (product_recommendation_data['probability_of_recommendation'] > 1) |
    (product_recommendation_data['probability_of_recommendation'].isna())
]

print(f"Invalid products recommendation data: {len(invalid_product_recommendation_data)}")



print(check_values_below_zero(customer_data))
print(check_values_below_zero(product_recommendation_data))

print(check_ratings(product_recommendation_data, 'average_rating_of_similar_products'))
print(check_ratings(product_recommendation_data, 'product_rating'))



In [ ]:
print(product_recommendation_data.head())

In [ ]:
print(customer_data.head())


In [ ]:
# Normalize list looking values to joined string values

def normalize_list_to_string(value):
    try:
         value_as_list = ast.literal_eval(value)
         if isinstance(value_as_list, list):
             values_lower = [
                 str(val).strip().lower() for val in value_as_list
             ]

             return ', '.join(values_lower)
    except (ValueError, SyntaxError):
        pass

    return value


def normalize_cols(df, cols):
    for col in cols:
        df[col] = df[col].apply(normalize_list_to_string)

    return df


normalize_cols(customer_data, ['browsing_history', 'purchase_history'])
normalize_cols(product_recommendation_data, ['similar_product_list'])

print(customer_data.head())
print('\n ------------------------------\n')
print(product_recommendation_data.head())


In [ ]:
product_recommendation_data.isna().sum()

In [ ]:
customer_data.isna().sum()

In [ ]:
# Customer count per segment

customer_per_segment = (
    customer_data
    .groupby('customer_segment')
    .agg(customer_count=('customer_id', 'count'))
    .reset_index()
    .sort_values('customer_count', ascending=False)
)
customer_per_segment.head()

In [ ]:
# Average order per customer segment

avg_order_per_segment = (
    customer_data
    .groupby('customer_segment')
    .agg(avg_order=('avg_order_value','mean'))
    .reset_index()
    .sort_values('avg_order', ascending=False)
)

avg_order_per_segment.head()

**New Visitor** segment has the highest average order value

In [ ]:
# Average order value per location

# avg_order_per_location = (
#     customer_data
#     .groupby('location')
#     .agg(avg_order=('avg_order_value','mean'))
#     .reset_index()
#     .sort_values('avg_order', ascending=False)
# )


# Using the .mean() method
avg_order_per_location = (
    customer_data
    .groupby('location')['avg_order_value']
    .mean()
    .reset_index(name='avg_order')
    .sort_values('avg_order', ascending=False)
)

avg_order_per_location.head()


**Mumbai** has the highest aveerage order value

In [ ]:
# Customer count per gender

customer_count_per_gender = (
    customer_data
    .groupby('gender')['customer_id']
    .count()
    .reset_index(name='customer_count')
    .sort_values('customer_count', ascending=False)
)

customer_count_per_gender.head()

In [ ]:
# Customer count by season and holiday

customer_count_per_season_holiday = (
    customer_data
    .groupby(['season', 'holiday'])['customer_id']
    .count()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

customer_count_per_season_holiday

**Autumn** has the highest-value customers

In [ ]:
customer_data.head()

In [ ]:
product_recommendation_data.head()

In [ ]:
# Average product price per category

avg_price_per_category = (
    product_recommendation_data
    .groupby('category')['price']
    .mean()
    .reset_index(name='avg_price')
    .sort_values('avg_price', ascending=False)
)

avg_price_per_category.head()

In [ ]:
# Avg product rating per category

rating_per_category = (
    product_recommendation_data
    .groupby('category')['product_rating']
    .mean()
    .reset_index(name='rating')
    .sort_values('rating', ascending=False)
)

rating_per_category.head()

In [ ]:
product_recommendation_data.columns

In [ ]:
# Average recommendation probability per brand

avg_recommendation_per_brand = (
    product_recommendation_data
    .groupby('brand')['probability_of_recommendation']
    .mean()
    .reset_index(name='avg_probability')
    .sort_values('avg_probability', ascending=False)
)

avg_recommendation_per_brand.head()

**Brand D** has the highest recommendation probability

In [ ]:
# Avg sentiment score

avg_sentiment = (
    product_recommendation_data
    .groupby('brand')['customer_review_sentiment_score']
    .mean()
    .reset_index(name='avg_sentiment')
    .sort_values('avg_sentiment', ascending=False)
)

avg_sentiment.head()

In [ ]:
# Number of products per category

products_per_category = (
    product_recommendation_data
    .groupby('category')['product_id']
    .nunique()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

products_per_category.head()

In [ ]:
# Highest avg rating per brand

top_brand_rating = (
    product_recommendation_data
    .groupby('brand')['product_rating']
    .mean()
    .reset_index(name='rating')
    .sort_values('rating', ascending=False)
)

top_brand_rating.head(1)

In [ ]:
# Highest avg price per category

category_avg_price = (
    product_recommendation_data
    .groupby('category')['price']
    .mean()
    .reset_index(name='avg_price')
    .sort_values('avg_price', ascending=False)
)

category_avg_price.head()

The category **Fashion** hast the highest average price

In [ ]:
# Product recommendation score table by brand and category

prod_recommendation_by_score = (
    product_recommendation_data
    .groupby(['brand', 'category'])['probability_of_recommendation']
    .mean()
    .reset_index(name='avg_recommendation_score')
    .sort_values('avg_recommendation_score', ascending=False)
)

prod_recommendation_by_score.head(25)

In [ ]:
# Brand summary table

brand_summary_table = (
    product_recommendation_data
    .groupby(['brand', 'category'])
    .agg(
        avg_product_rating=('product_rating', 'mean'),
        avg_sentiment=('customer_review_sentiment_score', 'mean'),
        avg_recommendation_probability=('probability_of_recommendation', 'mean'),
        avg_price=('price', 'mean'),
        product_count=('product_id', 'nunique'),
    )
    .reset_index()
)

brand_summary_table.head()

In [ ]:
product_recommendation_data.head(

)

In [ ]:
# High potential table

avg_sentiment_score = product_recommendation_data['customer_review_sentiment_score'].mean()
avg_probability_of_recommendation = product_recommendation_data['probability_of_recommendation'].mean()

high_potential = product_recommendation_data.loc[
    (product_recommendation_data['product_rating'] >=  4) &
    (product_recommendation_data['customer_review_sentiment_score'] >= avg_sentiment_score) &
    (product_recommendation_data['probability_of_recommendation'] >= avg_probability_of_recommendation ),
    [
        "product_id",
        "brand",
        "category",
        "subcategory",
        "price",
        "product_rating",
        "customer_review_sentiment_score",
        "probability_of_recommendation",
        "season",
        "holiday",
        "geographical_location"
    ]
]

print(high_potential.shape)
print(high_potential.head())


In [ ]:
print(customer_data)


In [ ]:
print(product_recommendation_data
      )

In [ ]:
# Recommendation targeting table

recommendation_target_table = (
    customer_data
    .merge(
        product_recommendation_data,
        on=['holiday', 'season'],
        how='inner'
    )
)

recommendation_target_table['recommendation_type'] = recommendation_target_table['probability_of_recommendation'] .apply(
    lambda x: 'high-priority' if x >= 0.75 else 'standard'
)


print(recommendation_target_table.shape)

In [ ]:
recommendation_target_table.head(20)


In [ ]:
# Best Segment Summary

best_segment_summary = (
    recommendation_target_table
    .groupby(['brand', 'customer_segment'])
    .agg(
        avg_order_value=('avg_order_value', 'mean'),
        number_of_customers=('customer_id', 'nunique'),
        avg_recommendation_probability=('probability_of_recommendation', 'mean'),
        best_location=('location', lambda x: x.mode().iloc[0]),
        best_season=('season', lambda x: x.mode().iloc[0]),
    )
    .reset_index()

)

best_segment_summary.head()

In [ ]:
best_segment_summary.shape


In [ ]:
# Final Brand Targeting

final_brand_targeting = (
    best_segment_summary
    .sort_values(['avg_order_value'], ascending=False)
    .groupby('brand')
    .head(1)
    .reset_index(drop=True)
)

final_brand_targeting.head()

In [ ]:
# Avg recommendation probability per category plot

avg_recommendation_per_category = (
    product_recommendation_data
    .groupby('category')['probability_of_recommendation']
    .mean()
    .reset_index(name='avg_recommendation_score')
    .sort_values('avg_recommendation_score', ascending=False)
)


plt.figure(figsize=(10, 6))
x = avg_recommendation_per_category['category']
y = avg_recommendation_per_category['avg_recommendation_score']

plt.bar(
    x,
    y,
)

plt.title("Average Recommendation Probability by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Average Recommendation Probability")
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
# Avg order value per customer segment plot

avg_order_per_segment.head()

x = avg_order_per_segment['customer_segment']
y = avg_order_per_segment['avg_order']

plt.figure(figsize=(10, 6))

plt.bar(
    x,
    y,
)

plt.title("Average Order Value per Customer Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Avg Order Value")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Product rating vs recommendation probability box plot

product_recommendation_data["rating_group"] = pd.cut(
    product_recommendation_data["product_rating"],
    bins=[0, 2, 3, 4, 5],
    labels=["0-2", "2-3", "3-4", "4-5"]
)

In [ ]:
data_to_plot = [
    product_recommendation_data.loc[
        product_recommendation_data["rating_group"] == group,
        "probability_of_recommendation"
    ]
    for group in product_recommendation_data["rating_group"].cat.categories
]

plt.figure(figsize=(10, 6))

plt.boxplot(
    data_to_plot,
    labels=product_recommendation_data["rating_group"].cat.categories
)

plt.title("Recommendation Probability by Product Rating Group")
plt.xlabel("Product Rating Group")
plt.ylabel("Recommendation Probability")
plt.tight_layout()

plt.show()